In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

load_dotenv()   # .env 파일에 저장되어있는 api 키 가져오기

True

In [ ]:
# 모델 설정
model = init_chat_model("openai:gpt-5.6-luna") 
#        # 챗봇 모델 초기화 #모델 파라미터 지정

# 벡터 저장소 설정
# 임베딩 및 저장
DB_PATH = "../data/k_ladder_2026"
# 경로 변수 - 실제경로

# 임베딩 모델 설정
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
#임베딩 모델 변수 - openai 임베딩 모델 초기화 함수 #모델 파라미터 지정

# 저장된 벡터 DB 가져오기
load_vs = Chroma(
    collection_name="k_ladder_2026",
    #DB 저장소 이름 설정 
    embedding_function=embeddings,
    #DB 임베딩 모델 기능 사용 - 임베딩 모델 변수 사용
    persist_directory=DB_PATH
    #지속적인 저장소 - db 변수 
)

In [3]:
# 검색기
retreiver_mmr = load_vs.as_retriever(search_type="mmr",
                                     search_kwargs={"k": 5, 
                                                    "fetch_k": 50,
                                                    "lambda_mult" : 0.25})

In [ ]:
# RAG 로 붙여보기
SYSTEM_PROMPT = """
너는 공공 정책 안내 도우미다.
아래 자료를 참고해서 답해라. 
참고 자료에 없으면 "자료에 없음" 이라고 말해라
정확한 자격, 금액, 기한은 공고 확인이 필요하다고 꼭 덧붙여라.
답 끝에 참고한 페이지 번호를 [p.60] 과 같은 형식으로 표시해라.
"""

# 검색 결과 문서를 받았을 때 메타데이터와 내용을 합쳐서 text 로 반환하는 함수 작성
def format_docs(docs):
    context = ""

    for doc in docs:
        context += f"[p.{doc.metadata['page']}] \n {doc.page_content} \n\n"
                        # 페이지 번호, 문서내용

    return context

In [ ]:
#확인용
chain = retreiver_mmr | format_docs
chain

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000002C1B5F4D7C0>, search_type='mmr', search_kwargs={'k': 5, 'fetch_k': 50, 'lambda_mult': 0.25})
| RunnableLambda(format_docs)

In [11]:
#확인용
print(chain.invoke("청년 월세 지원 정책 찾아줘"))

[p.39] 
 037모두의 정책 K-희망사다리 2026
여성청소년 
생리용품 지원
지원대상 	 • 	기초생활수급(생계·의료·주거·교육급여),	법정차상위계층,	한부모가족	지원	
대상	가구의	9~24세	여성청소년
핵심내용 	 •	 여성청소년	생리용품	바우처	지원(월	1만	4,000원),	국민행복카드로	구매
 •9세가	되는	해의	1월	1일부터	24세가	끝나는	해의	12월	31일까지	지원	
이용방법 	 •	 온라인	신청:	복지로(www.bokjiro.go.kr)	또는	모바일	앱
	 •방문	신청:	읍·면·동	주민센터	및	행정복지센터	
  ※  지원대상 결정 전·후 청소년 본인 또는 신청인(바우처 신청서상의 신청인) 명의의 국민
행복카드를 발급받아야 사용 가능 
문의처	• 성평등가족부	청소년정책과(☎02-2100-6242)
	 •한국사회보장정보원(☎1566-3232)
	 •읍·면·동	주민센터	및	행정복지센터		
02-2100-6242
성평등가족부 청소년정책과
1만  4,000원
월	지원금 

[p.14] 
 청년미래적금
 1600-5500
금융위원회
012따뜻한 동행 모두가 행복한 사회 - 2026년 신규 민생지원 제도
지원대상 	 •	 일정	소득	이하	만	19~34세	청년(병역	최대	6년	인정)
	 	 - 		일반형:	개인	소득	6,000만	원	이하	소득자	또는	연	매출	3억	원	이하	소상공인	
중	가구	중위소득	200%	이하
	 	 - 		우대형:	개인소득	3,600만	원	이하	중소기업	재직자	또는	연	매출	1억	원	
이하	소상공인	중	가구	중위소득	150%	이하
   ※  일반형 요건을 충족하는 중소기업 신규 재직자는 우대형 분류
핵심내용 	 • 	만기	3년
	 •	납입액(월	50만	원	한도)에	대한	정부기여금	지원(일반형	6%,	우대형	12%)	
및	이자소득	비과세
  ※  개인소득 6,000~7,500만 원 이하는 이자소득 비과세만 부여
이용방법 	 • 	신청	기간:	2026년	6월	이후(추후	안내	예정)
	 •신청	방법:	비대면	가입	신청(추후	안내	예정)


In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# rag_prompt 완성해보기
rag_prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('human', "참고자료\n{context} \n질문{question}")
])

                            
rag_chain = (
    {"context" : ( retreiver_mmr | format_docs ), 
     "question" :  RunnablePassthrough()} # question 은 검색기를 거쳐서 문서찾아서 context 키값의 벨류 
    | rag_prompt  # question 은 그대로 prompt 에 들어가야
    | model
    | StrOutputParser()
    )

print(rag_chain.invoke("청년 월세 지원 정책 찾아줘"))

청년 월세 지원 정책에 관한 내용은 제공된 자료에 없습니다. **자료에 없음**

정확한 지원 자격, 지원 금액, 신청 기간·방법은 해당 연도 및 지자체의 공식 공고를 반드시 확인해야 합니다.

참고 페이지: [p.14][p.39][p.89][p.240]


In [12]:
# 구조화된 출력 받기
from pydantic import BaseModel, Field

class AnswerStyle(BaseModel):
    answer : str = Field(description="최종 답변")
    source : str = Field(description="출처")

structured_model = model.with_structured_output(AnswerStyle, method="json_schema")
                            
rag_chain = (
    {"context" : ( retreiver_mmr | format_docs ), "question" :  RunnablePassthrough()} # question 은 검색기를 거쳐서 문서찾아서 context 키값의 벨류 
    | rag_prompt  # question 은 그대로 prompt 에 들어가야
    | structured_model
    )

result = rag_chain.invoke("청년 월세 지원 정책 찾아줘")

In [13]:
result.model_dump()

{'answer': '청년 월세 지원 정책은 제공된 참고자료에 없습니다. 정확한 지원 자격, 지원 금액, 신청 기간·기한은 해당 연도와 지역의 공식 공고를 반드시 확인해야 합니다. 거주지 관할 읍·면·동 주민센터나 복지로(www.bokjiro.go.kr)에서 확인해 보세요.',
 'source': '제공된 참고자료의 청년 정책 관련 페이지를 확인했으나 청년 월세 지원 내용은 없음 [p.14, p.89, p.94, p.240]'}

In [14]:
model.invoke("신대방 삼거리 맛집 알려줘")

AIMessage(content='신대방삼거리에서 찾기 좋은 곳 몇 군데 추천드릴게요. 다만 지점·영업시간은 방문 전 지도에서 확인해 주세요.\n\n### 1. 온정돈까스\n- **메뉴:** 돈까스, 매운 돈까스\n- **추천 포인트:** 이 동네에서 가장 유명한 메뉴 중 하나예요. 매운맛은 꽤 강한 편이라 처음이면 기본 돈까스부터 추천합니다.\n- **추천 상황:** 든든한 한 끼, 매운 음식 좋아할 때\n\n### 2. 백채김치찌개 신대방삼거리점\n- **메뉴:** 돼지고기 김치찌개, 계란말이\n- **추천 포인트:** 2~3명이 가서 찌개와 계란말이 같이 먹기 좋고, 가성비가 괜찮은 편입니다.\n- **추천 상황:** 밥다운 밥이 먹고 싶을 때\n\n### 3. 성대시장 먹거리\n- **메뉴:** 분식, 족발, 순대, 칼국수, 닭강정 등\n- **추천 포인트:** 신대방삼거리에서 조금 걸어도 괜찮다면 성대시장 쪽이 선택지가 많습니다. 한 곳보다 시장 구경하며 골라 먹기 좋아요.\n- **추천 상황:** 여러 명이 각자 다른 메뉴를 먹고 싶을 때\n\n### 4. 신대방삼거리역 주변 고깃집\n- **메뉴:** 삼겹살, 목살, 돼지갈비\n- **추천 포인트:** 역 주변에 고깃집이 여러 곳 있어서 회식이나 저녁 술자리로 무난합니다. 리뷰 수와 최근 후기가 많은 곳을 고르면 실패 확률이 낮아요.\n\n### 5. 보라매공원 방향 카페·식당\n- **추천 포인트:** 식사 후 산책하거나 조용히 커피 마시기 좋습니다. 신대방삼거리에서 보라매공원 쪽으로 가면 식당과 카페 선택지가 더 넓어져요.\n\n**한 곳만 고르면:**  \n- 돈까스 → **온정돈까스**  \n- 찌개·밥 → **백채김치찌개**  \n- 여러 메뉴·시장 분위기 → **성대시장**\n\n원하시면 **혼밥 / 데이트 / 술집 / 가성비 / 매운맛** 중 원하는 스타일로 더 좁혀서 추천해드릴게요.', additional_kwargs={'refusal': None}, response_metadata=

In [15]:
structured_model.invoke("신대방 삼거리 맛집 알려줘").model_dump()

{'answer': '신대방삼거리역 주변이라면 아래처럼 골라보세요. 다만 매장 이전·영업시간은 바뀔 수 있으니 방문 전 지도 앱에서 확인하는 게 좋습니다.\n\n- **가볍게 먹기:** 성대시장 쪽 분식, 순대, 족발, 칼국수집 — 역에서 도보 5~10분, 메뉴 선택지가 많아요.\n- **든든한 식사:** 역 주변의 순댓국·감자탕·제육백반집 — 혼밥하기 좋고 가격대도 무난합니다.\n- **고기:** 신대방삼거리역 1~2번 출구 안쪽 골목의 삼겹살·돼지갈비집 — 2~4명이 가기 좋아요.\n- **술자리:** 성대시장 입구와 역 뒤쪽 골목의 닭구이·곱창·이자카야 매장 — 1차 식사와 2차 술 모두 해결하기 좋습니다.\n- **산책 겸 식사:** 보라매공원 방향으로 이동하면 국수, 돈가스, 카페가 많아 식후 산책하기 좋아요.\n\n개인적으로는 **성대시장 구경 → 시장 안 식사 → 보라매공원 산책** 코스를 추천합니다. 원하는 메뉴가 고기·술집·혼밥·데이트 중 무엇인지 알려주시면 신대방삼거리역 기준으로 더 구체적인 가게를 골라드릴게요.',
 'source': '실시간 지도·영업 여부를 조회하지 않은 일반적인 지역 추천입니다. 방문 전 네이버지도·카카오맵에서 최신 정보를 확인하세요.'}

In [10]:
print(type(model))
print(rag_chain)

<class 'langchain_openai.chat_models.base.ChatOpenAI'>
first={
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000002C1B5F4D7C0>, search_type='mmr', search_kwargs={'k': 5, 'fetch_k': 50, 'lambda_mult': 0.25})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
} middle=[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='\n너는 공공 정책 안내 도우미다.\n아래 자료를 참고해서 답해라. \n참고 자료에 없으면 "자료에 없음" 이라고 말해라\n정확한 자격, 금액, 기한은 공고 확인이 필요하다고 꼭 덧붙여라.\n답 끝에 참고한 페이지 번호를 [p.60] 과 같은 형식으로 표시해라.\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='참고자료\n{context} \n질문{question}'), additional_kwargs={})]), _ChatModelBinding(bound=

In [ ]:
# 초등학생도 풀 수 있는 RAG 질문 연습
practice_questions = [
    '이 문서는 어떤 정책을 설명하고 있나요?',
    '청년 월세 지원은 누구를 위한 제도인가요?',
    '신청할 때 확인해야 할 것은 무엇인가요?',
    '지원 내용에서 꼭 기억할 숫자는 무엇인가요?',
    '더 자세히 물어보려면 어디에 문의하면 되나요?',
]

for number, question in enumerate(practice_questions, start=1):
    print(f'문제 {number}: {question}')
    answer = rag_chain.invoke(question)
    print(answer)
    print('-' * 60)

## 연습 문제 확인표

각 답변을 보고 다음 세 가지를 확인해 보세요.

1. 질문에 바로 답했나요?
2. 문서에 없는 내용을 상상해서 말하지 않았나요?
3. 답변에 참고 페이지가 표시되었나요?

답변이 이상하면 검색 결과 개수 `k`를 3 또는 5로 바꾸고 다시 실행해 보세요.


In [ ]:
# 내가 만든 질문 하나로 다시 테스트하기
my_question = '청년 월세 지원을 받으려면 무엇을 먼저 확인해야 하나요?'
my_answer = rag_chain.invoke(my_question)
print(my_answer)

## 가장 간단한 RAG 실습

질문을 하면 관련 문서를 찾고, 그 문서를 읽은 AI가 답변합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

simple_prompt = ChatPromptTemplate.from_template("""
아래 문서만 참고해서 질문에 답해줘.
문서에 답이 없으면 '문서에서 찾지 못했어요'라고 말해줘.

문서:
{context}

질문:
{question}
""")

simple_chain = (
    {
        'context': retreiver_mmr | format_docs,
        'question': RunnablePassthrough(),
    }
    | simple_prompt
    | model
    | StrOutputParser()
)

simple_chain.invoke('청년 월세 지원은 누구를 위한 제도인가요?')